# ArduMedics Notebook 02: YOLOv8s-Pose Fall Detection Training

---

**Project:** ArduMedics — AI-Powered Healthcare Robot  
**Notebook:** 02 / 06  
**Previous:** [Notebook 01 — YOLOv8n-Pose (Nano)](./ArduMedics_01_YOLOv8n_Pose_Fall_Detection_Training.ipynb)  
**Next:** [Notebook 03 — YOLOv8m-Pose (Medium)](./ArduMedics_03_YOLOv8m_Pose_Fall_Detection_Training.ipynb)  

---

## Objective

Train the **YOLOv8s-Pose (Small)** model — the mid-tier architecture that balances accuracy and speed for real-time fall detection on the ArduMedics healthcare robot.

## Key Differences from Notebook 01 (Nano)

| Parameter | NB01 (Nano) | **NB02 (Small)** |
|-----------|-------------|------------------|
| Model | YOLOv8n-pose | **YOLOv8s-pose** |
| Parameters | 3.2M | **11.2M** |
| Epochs | 150 | **100** |
| Batch Size | 16 | **8** |
| Mixup | 0.1 | **0.15** |
| Export Formats | NCNN, TFLite, ONNX | **NCNN, TFLite (INT8), ONNX** |

## Training Strategy

- **Architecture:** YOLOv8s-pose — 11.2M parameters for improved keypoint accuracy
- **Optimizer:** AdamW with cosine annealing learning rate schedule
- **Augmentation:** Mosaic, MixUp (0.15), Copy-Paste (0.3), geometric & color transforms
- **Early Stopping:** Patience of 30 epochs to prevent overfitting
- **Export:** NCNN (edge deployment), TFLite INT8 (mobile), ONNX (cross-platform)

## Datasets

### Roboflow Pose Datasets
1. **Falling Pose Estimation** (635 images, 1-class: person→fall) — [Roboflow](https://universe.roboflow.com/humna-pose-data/falling-pose-estimation)
2. **YOLOv8-Pose Fall Detection** (474 images, 2-class: fall/not-fallen) — [Roboflow](https://universe.roboflow.com/yolo-xvnzo/yolov8-pose-utovc)

> **NOTE:** Dataset 3 (Nafzzan/falling-pose-estimation-0xme8) was removed — it is a pixel-identical duplicate of Dataset 1 (same 635 images).

### Kaggle Datasets
4. **UR Fall Detection** — [Kaggle](https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset)
5. **Fall Detection Images** — [Kaggle](https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset)
6. **Le2i Fall Dataset** — [Kaggle](https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia)
7. **Multiple Cameras Fall** — [Kaggle](https://www.kaggle.com/datasets/soumicksarker/multiple-cameras-fall-dataset)
8. **Fall Video Dataset** — [Kaggle](https://www.kaggle.com/datasets/payutch/fall-video-dataset)

## Estimated Time

~10–11 hours on NVIDIA T4 GPU (Kaggle)

---

## Step 1: Environment Setup

Install required packages and import all libraries. Set the random seed for reproducibility across all experiments.

In [1]:
# ============================================================
# OUTPUT MANAGEMENT UTILITIES (ArduMedics Standard)
# Suppresses noisy output while keeping important logs
# ============================================================

import os, sys, warnings, contextlib, io

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'
os.environ['OPENCV_VIDEOIO_DEBUG'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['FLAGS_logtostderr'] = '0'
os.environ['GLOG_minloglevel'] = '3'
os.environ['YOLO_AUTODOWNLOAD'] = '1'

class suppress_output:
    def __enter__(self):
        self._orig = sys.stdout
        sys.stdout = io.StringIO()
        return self
    def __exit__(self, *a):
        sys.stdout = self._orig
        return False

_real_stdout = sys.stdout
def important_print(msg, end='\n'):
    _real_stdout.write(str(msg) + end)
    _real_stdout.flush()

print('[ArduMedics] Output management loaded')


[ArduMedics] Output management loaded


In [2]:
# Notebook: 02 | Step: 1 of 8 — Environment Setup
# After this: Download Roboflow pose datasets

!pip install -qq ultralytics roboflow opencv-python-headless

import os
import json
import random
import shutil
import glob
import yaml
import cv2
cv2.setLogLevel(0)
import numpy as np
from pathlib import Path
from datetime import datetime

from ultralytics import YOLO

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Directory constants
WORK_DIR = Path("/kaggle/working")
DATASETS_DIR = WORK_DIR / "datasets"
RUNS_DIR = WORK_DIR / "runs"
EXPORT_DIR = WORK_DIR / "exports"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment ready. SEED={SEED}")
print(f"Working directory: {WORK_DIR}")
print(f"Ultralytics version: {__import__('ultralytics').__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 77.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo set

## Step 2: Download Roboflow Pose Datasets

Download the same 2 Roboflow pose-format datasets used in Notebook 01. These come pre-annotated with keypoints in YOLO pose format, so they can be used directly without relabeling.

### Dataset Source Priority
1. **Primary** — Unified dataset already in `/kaggle/working/datasets/ardumedics_unified_pose/` (from NB01)
2. **Secondary** — Recursive scan of ALL of `/kaggle/input/` for `data.yaml` (copies to writable dir)
3. **Fallback** — Roboflow SDK download (requires API key)

> **ℹ️ The code recursively scans `/kaggle/input/` using `os.walk` to find any `data.yaml` files.** It matches datasets by name patterns (e.g., `falling_pose_estimation`, `yolov8-pose`, `yolo-xvnzo`) and **copies them to `/kaggle/working/datasets/`** (writable) — never using `/kaggle/input/` as a working directory.

**Class Mapping Strategy:** The two datasets use different class schemes:
- Dataset 1 (Falling Pose Estimation): `nc=1`, class 0=`person` → remaps to class 0=`fall` (all images depict falling poses)
- Dataset 2 (YOLOv8-Pose Fall Detection): `nc=2`, class 0=`fall`, class 1=`not-fallen` → used as-is
- **Unified scheme:** `nc=2`, names=`['fall', 'not-fallen']`

> **Note:** Replace `ROBOFLOW_API_KEY` with your actual key (only needed for Fallback). The unified dataset from NB01 may already exist — we check before re-downloading.

In [3]:
# Notebook: 02 | Step: 2 of 8 — Download Roboflow Pose Datasets
# After this: Download Kaggle datasets and extract frames

# ROBUST DATASET LOADING: 4-tier approach
#   1. Check for unified dataset from NB01
#   2. Recursively scan /kaggle/input/ for data.yaml
#   3. Download from Kaggle dataset (nishatfifa/ardumedics-roboflow-pose-datasets)
#   4. Fallback: Roboflow SDK (requires API key)

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY_HERE")

# Check if unified dataset already exists from NB01
unified_dir = DATASETS_DIR / "ardumedics_unified_pose"
_need_download = False

# ── ROBUST: Recursively find ALL data.yaml in /kaggle/input/ ──
DATASET1_PATTERNS = ['falling_pose_estimation', 'falling-pose-estimation',
                     'Falling pose estimation', 'humna']
DATASET2_PATTERNS = ['yolov8_pose_fall', 'yolov8-pose',
                     'yolov8-pose.v1i', 'yolov8-pose.v1xme8', 'yolo-xvnzo']

found_ds1 = None
found_ds2 = None

# Check for existing unified dataset first
if unified_dir.exists() and (unified_dir / "data.yaml").exists():
    print(f"Unified dataset already exists at {unified_dir}")
    print("Skipping Roboflow download.")
elif Path("/kaggle/input").exists():
    # Recursively scan for data.yaml
    for root, dirs, files in os.walk("/kaggle/input"):
        if 'data.yaml' in files:
            root_lower = root.lower()
            if any(p.lower() in root_lower for p in DATASET1_PATTERNS):
                found_ds1 = root
                print(f"  Found Dataset 1 at: {root}")
            elif any(p.lower() in root_lower for p in DATASET2_PATTERNS):
                found_ds2 = root
                print(f"  Found Dataset 2 at: {root}")
            else:
                if found_ds1 is None:
                    found_ds1 = root
                    print(f"  Found unknown dataset, assigning as Dataset 1: {root}")
                elif found_ds2 is None:
                    found_ds2 = root
                    print(f"  Found unknown dataset, assigning as Dataset 2: {root}")

    # Copy found datasets to writable directory
    if found_ds1:
        dst = DATASETS_DIR / "falling_pose_estimation"
        if not dst.exists():
            shutil.copytree(found_ds1, str(dst))
        print(f"  Dataset 1 → falling_pose_estimation/")
    
    if found_ds2:
        dst = DATASETS_DIR / "yolov8_pose_fall"
        if not dst.exists():
            shutil.copytree(found_ds2, str(dst))
        print(f"  Dataset 2 → yolov8_pose_fall/")
    
    if found_ds1 or found_ds2:
        print("Datasets copied to writable directory!")
    else:
        _need_download = True
else:
    _need_download = True

# ── Tier 3: Try downloading from Kaggle dataset ──
if _need_download:
    print("[Tier 3] Trying Kaggle dataset download...")
    try:
        kaggle_ds_dir = DATASETS_DIR / "ardumedics_roboflow_temp"
        kaggle_ds_dir.mkdir(parents=True, exist_ok=True)
        !kaggle datasets download -d nishatfifa/ardumedics-roboflow-pose-datasets -p {kaggle_ds_dir} --unzip
        
        # Scan the downloaded directory for data.yaml
        for root, dirs, files in os.walk(str(kaggle_ds_dir)):
            if 'data.yaml' in files:
                root_lower = root.lower()
                if any(p.lower() in root_lower for p in DATASET1_PATTERNS) and found_ds1 is None:
                    found_ds1 = root
                    print(f"  Found Dataset 1 at: {root}")
                elif any(p.lower() in root_lower for p in DATASET2_PATTERNS) and found_ds2 is None:
                    found_ds2 = root
                    print(f"  Found Dataset 2 at: {root}")
                elif found_ds1 is None:
                    found_ds1 = root
                    print(f"  Found unknown dataset, assigning as Dataset 1: {root}")
                elif found_ds2 is None:
                    found_ds2 = root
                    print(f"  Found unknown dataset, assigning as Dataset 2: {root}")
        
        # Copy found datasets
        if found_ds1:
            dst = DATASETS_DIR / "falling_pose_estimation"
            if not dst.exists():
                shutil.copytree(found_ds1, str(dst))
            print(f"  Dataset 1 → falling_pose_estimation/")
        
        if found_ds2:
            dst = DATASETS_DIR / "yolov8_pose_fall"
            if not dst.exists():
                shutil.copytree(found_ds2, str(dst))
            print(f"  Dataset 2 → yolov8_pose_fall/")
        
        if found_ds1 or found_ds2:
            print("Datasets downloaded from Kaggle and copied!")
            _need_download = False
            # Clean up temp directory
            shutil.rmtree(str(kaggle_ds_dir), ignore_errors=True)
        else:
            print("  Kaggle download succeeded but no data.yaml found inside.")
            shutil.rmtree(str(kaggle_ds_dir), ignore_errors=True)
    except Exception as e:
        print(f"  Kaggle dataset download failed: {e}")
        shutil.rmtree(str(DATASETS_DIR / "ardumedics_roboflow_temp"), ignore_errors=True)

# ── Tier 4: Fallback — Roboflow SDK download ──
if _need_download:
    print("[Fallback] Downloading from Roboflow SDK...")
    if ROBOFLOW_API_KEY == "YOUR_ROBOFLOW_API_KEY_HERE":
        print("=" * 60)
        print("ERROR: No datasets found and Roboflow API key not set!")
        print("=" * 60)
        print("To fix this, do ONE of:")
        print("  1. Add 'nishatfifa/ardumedics-roboflow-pose-datasets' as")
        print("     a Kaggle dataset input to this notebook")
        print("  2. Set ROBOFLOW_API_KEY environment variable")
        print("  3. Upload a ZIP containing falling_pose_estimation/ and")
        print("     yolov8_pose_fall/ to Kaggle and add as input")
        print("=" * 60)
        raise ValueError("No datasets available and Roboflow API key not configured")
    
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    # Dataset 1: Falling Pose Estimation (635 images)
    print("\n[1/2] Downloading Falling Pose Estimation...")
    project1 = rf.workspace("humna-pose-data").project("falling-pose-estimation")
    version1 = project1.version(2)
    with suppress_output():
        dataset1 = version1.download("yolov8", location=str(DATASETS_DIR / "falling_pose_estimation"))
    print(f"  -> Saved to: {DATASETS_DIR / 'falling_pose_estimation'}")
    
    # Dataset 2: YOLOv8-Pose Fall Detection (474 images)
    print("\n[2/2] Downloading YOLOv8-Pose Fall Detection...")
    project2 = rf.workspace("yolo-xvnzo").project("yolov8-pose-utovc")
    version2 = project2.version(3)
    with suppress_output():
        dataset2 = version2.download("yolov8", location=str(DATASETS_DIR / "yolov8_pose_fall"))
    print(f"  -> Saved to: {DATASETS_DIR / 'yolov8_pose_fall'}")

    print("\nBoth Roboflow datasets downloaded successfully!")

  Found Dataset 1 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/falling_pose_estimation
  Found Dataset 2 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/yolov8_pose_fall
  Dataset 1 → falling_pose_estimation/
  Dataset 2 → yolov8_pose_fall/
Datasets copied to writable directory!


## Step 3: Download Kaggle Datasets & Extract Frames

Download the Kaggle datasets. Some contain video files rather than images — we extract frames at a configurable rate. Non-pose-format annotations will be handled in Step 4.

### Kaggle Datasets
- **UR Fall Detection** — Image-based fall detection dataset
- **Fall Detection Images** — Image-based dataset
- **Le2i Fall Dataset** — Video-based, requires frame extraction
- **Multiple Cameras Fall** — Multi-camera video dataset
- **Fall Video Dataset** — Video-based fall dataset

In [4]:
# Notebook: 02 | Step: 3 of 8 — Download Kaggle Datasets & Extract Frames
# After this: Auto-label Kaggle frames using YOLOv8n-pose

# Skip if unified dataset already exists from NB01
if unified_dir.exists() and (unified_dir / "data.yaml").exists():
    print(f"Unified dataset already exists at {unified_dir}. Skipping Kaggle download.")
else:
    FRAME_INTERVAL = 5  # Extract every 5th frame from videos
    kaggle_frames_dir = DATASETS_DIR / "kaggle_frames"
    kaggle_frames_dir.mkdir(parents=True, exist_ok=True)

    def extract_frames_from_videos(video_dir, output_dir, interval=5):
        """Extract frames from all video files in a directory."""
        frame_count = 0
        video_extensions = {'.avi', '.mp4', '.mov', '.mkv', '.wmv'}
        
        for video_path in Path(video_dir).rglob('*'):
            if video_path.suffix.lower() in video_extensions:
                cap = cv2.VideoCapture(str(video_path))
                frame_idx = 0
                saved_idx = 0
                
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break
                    if frame_idx % interval == 0:
                        frame_name = f"{video_path.stem}_frame{saved_idx:05d}.jpg"
                        cv2.imwrite(str(output_dir / frame_name), frame)
                        saved_idx += 1
                    frame_idx += 1
                cap.release()
                frame_count += saved_idx
        
        return frame_count

    # Dataset 4: UR Fall Detection (images)
    print("[4/8] UR Fall Detection...")
    ur_dir = DATASETS_DIR / "ur_fall_detection"
    if not ur_dir.exists():
        !kaggle datasets download -d shahliza27/ur-fall-detection-dataset -p {ur_dir} --unzip
    ur_img_count = len(list(ur_dir.rglob('*.jpg'))) + len(list(ur_dir.rglob('*.png')))
    print(f"  -> {ur_img_count} images found")

    # Dataset 5: Fall Detection Images (images)
    print("[5/8] Fall Detection Images...")
    fall_img_dir = DATASETS_DIR / "fall_detection_images"
    if not fall_img_dir.exists():
        !kaggle datasets download -d uttejkumarkandagatla/fall-detection-dataset -p {fall_img_dir} --unzip
    fi_count = len(list(fall_img_dir.rglob('*.jpg'))) + len(list(fall_img_dir.rglob('*.png')))
    print(f"  -> {fi_count} images found")

    # Dataset 6: Le2i Fall Dataset (videos -> extract frames)
    # DISK AWARE: Le2i is ~9.4 GB. Kaggle /kaggle/working has 20 GB hard limit.
    # Le2i download+unzip temporarily needs ~18.8 GB (9.37 GB zip + 9.37 GB
    # unzipped), exceeding Kaggle capacity. Force-skip via 25 GB threshold
    # (impossible on Kaggle). Le2i is optional - Roboflow datasets suffice.
    # ⚠️ DISK AWARE: Le2i is ~9.4 GB. Skip if disk space is low.
    disk_free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
    print("[6/8] Le2i Fall Dataset...")
    le2i_dir = DATASETS_DIR / "le2i_fall"
    if disk_free_gb > 25 and not le2i_dir.exists():
        !kaggle datasets download -d tuyenldvn/falldataset-imvia -p {le2i_dir} --unzip
    elif disk_free_gb <= 25:
        print("  SKIPPED: Insufficient disk space for Le2i (needs ~20 GB free for download+unzip)")
    le2i_frames = kaggle_frames_dir / "le2i_frames"
    le2i_frames.mkdir(exist_ok=True)
    if le2i_dir.exists():
        le2i_count = extract_frames_from_videos(le2i_dir, le2i_frames, FRAME_INTERVAL)
        print(f"  -> {le2i_count} frames extracted")
        # Free ~9 GB: delete raw video files after extracting frames
        shutil.rmtree(str(le2i_dir), ignore_errors=True)
        print("  Deleted raw Le2i videos to free disk space")
    else:
        le2i_count = 0
        print("  No Le2i data to process")

    # Recalculate disk space after Le2i processing
    disk_free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
    print(f"  Disk free: {disk_free_gb:.1f} GB")
    # Dataset 7: Multiple Cameras Fall (videos -> extract frames)
    print("[7/8] Multiple Cameras Fall...")
    mcf_dir = DATASETS_DIR / "multiple_cameras_fall"
    if disk_free_gb > 25 and not mcf_dir.exists():
        !kaggle datasets download -d soumicksarker/multiple-cameras-fall-dataset -p {mcf_dir} --unzip
    elif disk_free_gb <= 25:
        print("  SKIPPED: Insufficient disk space for MCF dataset (needs ~20 GB free)")
    mcf_frames = kaggle_frames_dir / "mcf_frames"
    mcf_frames.mkdir(exist_ok=True)
    if mcf_dir.exists():
        mcf_count = extract_frames_from_videos(mcf_dir, mcf_frames, FRAME_INTERVAL)
        print(f"  -> {mcf_count} frames extracted")
        # Free disk: delete raw video files after extracting frames
        shutil.rmtree(str(mcf_dir), ignore_errors=True)
        print("  Deleted raw MCF videos to free disk space")
    else:
        mcf_count = 0
        print("  No MCF data to process")

    # Recalculate disk space after MCF processing
    disk_free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
    print(f"  Disk free: {disk_free_gb:.1f} GB")
    # Dataset 8: Fall Video Dataset (videos -> extract frames)
    print("[8/8] Fall Video Dataset...")
    fvd_dir = DATASETS_DIR / "fall_video_dataset"
    if disk_free_gb > 25 and not fvd_dir.exists():
        !kaggle datasets download -d payutch/fall-video-dataset -p {fvd_dir} --unzip
    elif disk_free_gb <= 25:
        print("  SKIPPED: Insufficient disk space for FVD dataset (needs ~20 GB free)")
    fvd_frames = kaggle_frames_dir / "fvd_frames"
    fvd_frames.mkdir(exist_ok=True)
    if fvd_dir.exists():
        fvd_count = extract_frames_from_videos(fvd_dir, fvd_frames, FRAME_INTERVAL)
        print(f"  -> {fvd_count} frames extracted")
        # Free disk: delete raw video files after extracting frames
        shutil.rmtree(str(fvd_dir), ignore_errors=True)
        print("  Deleted raw FVD videos to free disk space")
    else:
        fvd_count = 0
        print("  No FVD data to process")

    # Clean up any leftover .zip files from kaggle downloads
    for zip_file in DATASETS_DIR.rglob('*.zip'):
        zip_file.unlink()
        print(f"  Deleted leftover zip: {zip_file.name}")

    print(f"\nKaggle datasets processed. Frames saved to: {kaggle_frames_dir}")

[4/8] UR Fall Detection...
Dataset URL: https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset
License(s): unknown
100%|██████████████████████████████████████| 4.18G/4.18G [03:02<00:00, 24.6MB/s]

  -> 11936 images found
[5/8] Fall Detection Images...
Dataset URL: https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset
License(s): ODbL-1.0
100%|██████████████████████████████████████| 49.7M/49.7M [00:03<00:00, 16.8MB/s]

  -> 485 images found
[6/8] Le2i Fall Dataset...
  SKIPPED: Insufficient disk space for Le2i (needs ~20 GB free for download+unzip)
  No Le2i data to process
  Disk free: 16.3 GB
[7/8] Multiple Cameras Fall...
  SKIPPED: Insufficient disk space for MCF dataset (needs ~20 GB free)
  No MCF data to process
  Disk free: 16.3 GB
[8/8] Fall Video Dataset...
  SKIPPED: Insufficient disk space for FVD dataset (needs ~20 GB free)
  No FVD data to process

Kaggle datasets processed. Frames saved to: /kaggle/working/datasets/kaggle_frames


## Step 4: Auto-Label Kaggle Frames

The Kaggle datasets don't include pose-format annotations. We use the pre-trained YOLOv8n-pose model to generate pseudo-labels (keypoints + bounding boxes) for all extracted frames. This is a common weak-supervision technique.

**Process:**
1. Load the pre-trained YOLOv8n-pose model
2. Run inference on each frame
3. Convert detections to YOLO pose annotation format
4. Save `.txt` label files alongside images

In [5]:
# Notebook: 02 | Step: 4 of 8 — Auto-Label Kaggle Frames
# After this: Merge all datasets into unified format

if unified_dir.exists() and (unified_dir / "data.yaml").exists():
    print(f"Unified dataset already exists at {unified_dir}. Skipping auto-labeling.")
else:
    # Load pre-trained YOLOv8n-pose for pseudo-labeling
    labeler = YOLO("yolov8n-pose.pt")
    print("Loaded YOLOv8n-pose for pseudo-labeling")

    CONF_THRESHOLD = 0.5  # Minimum confidence for pseudo-labels

    def auto_label_images(image_dir, label_dir, model, conf=0.5):
        """
        Run pose estimation on images and save YOLO pose-format labels.
        
        YOLO pose label format per line:
        class_id cx cy w h kp1_x kp1_y kp1_v ... kp17_x kp17_y kp17_v
        All values normalized to [0, 1].
        """
        image_dir = Path(image_dir)
        label_dir = Path(label_dir)
        label_dir.mkdir(parents=True, exist_ok=True)
        
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
        image_files = sorted([
            f for f in image_dir.rglob('*') if f.suffix.lower() in image_extensions
        ])
        
        labeled_count = 0
        
        for img_path in image_files:
            results = model(str(img_path), verbose=False, conf=conf)
            
            if len(results) == 0 or results[0].keypoints is None:
                continue
            
            result = results[0]
            
            # Get image dimensions for normalization
            img_h, img_w = result.orig_shape[:2]
            
            label_lines = []
            boxes = result.boxes
            keypoints = result.keypoints
            
            for i in range(len(boxes)):
                # Bounding box (normalized xywh)
                box = boxes.xywhn[i].cpu().numpy()
                cls_id = int(boxes.cls[i].cpu().numpy())
                
                # Keypoints (normalized xy + visibility)
                kpts = keypoints.xyn[i].cpu().numpy()  # shape: (17, 3)
                
                # Build label line: class cx cy w h kp1_x kp1_y kp1_v ...
                line_parts = [cls_id] + box.tolist()
                for kp in kpts:
                    line_parts.extend(kp.tolist())  # x, y, visibility
                
                label_lines.append(" ".join([str(v) for v in line_parts]))
            
            if label_lines:
                label_path = label_dir / (img_path.stem + ".txt")
                with open(label_path, 'w') as f:
                    f.write("\n".join(label_lines))
                labeled_count += 1
        
        return labeled_count

    # Auto-label all Kaggle frame directories
    kaggle_frames_dir = DATASETS_DIR / "kaggle_frames"
    total_labeled = 0
    
    for frame_subdir in sorted(kaggle_frames_dir.iterdir()):
        if frame_subdir.is_dir():
            label_subdir = Path(str(frame_subdir).replace("_frames", "_labels"))
            print(f"Labeling {frame_subdir.name}...")
            count = auto_label_images(frame_subdir, label_subdir, labeler, CONF_THRESHOLD)
            print(f"  -> {count} images labeled")
            total_labeled += count
    
    # Also label non-frame Kaggle image directories
    for img_dataset in ["ur_fall_detection", "fall_detection_images"]:
        img_dir = DATASETS_DIR / img_dataset
        if img_dir.exists():
            label_dir = DATASETS_DIR / f"{img_dataset}_labels"
            print(f"Labeling {img_dataset}...")
            count = auto_label_images(img_dir, label_dir, labeler, CONF_THRESHOLD)
            print(f"  -> {count} images labeled")
            total_labeled += count

    print(f"\nTotal images auto-labeled: {total_labeled}")

Loaded YOLOv8n-pose for pseudo-labeling
Labeling fvd_frames...
  -> 0 images labeled
Labeling le2i_frames...
  -> 0 images labeled
Labeling mcf_frames...
  -> 0 images labeled
Labeling ur_fall_detection...
  -> 9396 images labeled
Labeling fall_detection_images...
  -> 455 images labeled

Total images auto-labeled: 9851


## Step 5: Merge All Datasets & Remap Classes

Merge all datasets (Roboflow + Kaggle auto-labeled) into a single unified dataset with standard YOLO pose format directory structure.

**Class Remapping:** Each Roboflow dataset uses a different class scheme. We remap all labels to the unified scheme `nc=2, names=['fall', 'not-fallen']` during the copy phase.

```
ardumedics_unified_pose/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
└── data.yaml
```

We use an 85/15 train/val split with SEED-based shuffling for reproducibility.

In [6]:
# Notebook: 02 | Step: 5 of 8 — Merge All Datasets
# After this: Create data.yaml and verify dataset integrity

if unified_dir.exists() and (unified_dir / "data.yaml").exists():
    print(f"Unified dataset already exists at {unified_dir}. Skipping merge.")
else:
    VAL_RATIO = 0.15
    
    unified_images_train = unified_dir / "images" / "train"
    unified_images_val = unified_dir / "images" / "val"
    unified_labels_train = unified_dir / "labels" / "train"
    unified_labels_val = unified_dir / "labels" / "val"
    
    for d in [unified_images_train, unified_images_val, unified_labels_train, unified_labels_val]:
        d.mkdir(parents=True, exist_ok=True)
    
    # --- Class remapping helper ---
    # Datasets use different class schemes that must be unified:
    #   Dataset 1 (falling_pose_estimation): nc=1, class 0="person" → 0="fall"
    #   Dataset 2 (yolov8_pose_fall): nc=2, class 0="fall", 1="not-fallen" → as-is
    # Unified scheme: nc=2, {0: 'fall', 1: 'not-fallen'}

    def remap_label_file(lbl_file, class_map, dest_dir, dest_name=None):
        """Remap class IDs in a YOLO pose label file according to class_map dict.

        Args:
            lbl_file: Path to source label file
            class_map: dict mapping old_class_id -> new_class_id (e.g. {0: 0}).
                      Empty dict means copy as-is (no remapping).
            dest_dir: Path to destination directory
            dest_name: Output filename (defaults to lbl_file.name)
        """
        if dest_name is None:
            dest_name = lbl_file.name
        dest_file = dest_dir / dest_name

        if not class_map:
            shutil.copy2(lbl_file, dest_file)
            return

        with open(lbl_file, 'r') as f:
            lines = f.readlines()

        remapped_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            old_cls = int(parts[0])
            if old_cls in class_map:
                parts[0] = str(class_map[old_cls])
                remapped_lines.append(' '.join(parts))
            # else: skip annotations with unrecognized class IDs

        with open(dest_file, 'w') as f:
            f.write('\n'.join(remapped_lines))

    # Collect all (image, label, class_map) triples
    pairs = []  # Each entry: (img_file, lbl_file, class_map_dict)

    # --- Roboflow datasets with class mapping config ---
    roboflow_datasets_config = [
        {
            "path": DATASETS_DIR / "falling_pose_estimation",
            "class_map": {0: 0},  # "person" → "fall" (all images show falling poses)
        },
        {
            "path": DATASETS_DIR / "yolov8_pose_fall",
            "class_map": {},  # Already uses unified scheme: 0="fall", 1="not-fallen"
        },
    ]

    for ds_config in roboflow_datasets_config:
        ds_dir = ds_config["path"]
        class_map = ds_config["class_map"]

        if not ds_dir.exists():
            print(f"Warning: {ds_dir} not found, skipping")
            continue

        # Roboflow datasets already have train/val/test splits
        for split in ["train", "valid", "test"]:
            img_dir = ds_dir / split / "images"
            lbl_dir = ds_dir / split / "labels"

            if not img_dir.exists():
                continue

            for img_file in img_dir.glob('*'):
                if img_file.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                    lbl_file = lbl_dir / (img_file.stem + ".txt")
                    if lbl_file.exists():
                        pairs.append((img_file, lbl_file, class_map))
    
    # --- Kaggle auto-labeled datasets ---
    kaggle_frame_dirs = sorted((DATASETS_DIR / "kaggle_frames").iterdir()) if (DATASETS_DIR / "kaggle_frames").exists() else []
    
    for frame_dir in kaggle_frame_dirs:
        if not frame_dir.is_dir():
            continue
        label_dir = Path(str(frame_dir).replace("_frames", "_labels"))
        if not label_dir.exists():
            continue
        
        for img_file in frame_dir.glob('*'):
            if img_file.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                lbl_file = label_dir / (img_file.stem + ".txt")
                if lbl_file.exists():
                    pairs.append((img_file, lbl_file, {}))  # No remapping needed
    
    # --- Kaggle image datasets with auto-labels ---
    for img_dataset in ["ur_fall_detection", "fall_detection_images"]:
        img_dir = DATASETS_DIR / img_dataset
        lbl_dir = DATASETS_DIR / f"{img_dataset}_labels"
        
        if not img_dir.exists() or not lbl_dir.exists():
            continue
        
        for img_file in img_dir.rglob('*'):
            if img_file.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                lbl_file = lbl_dir / (img_file.stem + ".txt")
                if lbl_file.exists():
                    pairs.append((img_file, lbl_file, {}))  # No remapping needed
    
    print(f"Total image-label pairs found: {len(pairs)}")
    
    # Shuffle and split
    random.seed(SEED)
    random.shuffle(pairs)
    
    val_count = int(len(pairs) * VAL_RATIO)
    val_pairs = pairs[:val_count]
    train_pairs = pairs[val_count:]
    
    print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)}")
    
    # Copy files to unified directory with unique names, applying class remapping
    def copy_pairs(pair_list, img_dest, lbl_dest):
        for img_file, lbl_file, class_map in pair_list:
            # Create unique filename to avoid collisions
            unique_name = f"{img_file.parent.stem}_{img_file.name}"
            shutil.copy2(img_file, img_dest / unique_name)
            lbl_dest_name = unique_name.replace(img_file.suffix, ".txt")
            remap_label_file(lbl_file, class_map, lbl_dest, dest_name=lbl_dest_name)
    
    print("Copying train pairs...")
    copy_pairs(train_pairs, unified_images_train, unified_labels_train)
    
    print("Copying val pairs...")
    copy_pairs(val_pairs, unified_images_val, unified_labels_val)
    
    print(f"\nUnified dataset created at: {unified_dir}")
    print(f"  Train images: {len(list(unified_images_train.glob('*')))}")
    print(f"  Val images: {len(list(unified_images_val.glob('*')))}")
    print(f"  Train labels: {len(list(unified_labels_train.glob('*')))}")
    print(f"  Val labels: {len(list(unified_labels_val.glob('*')))}")

Total image-label pairs found: 10978
Train: 9332 | Val: 1646
Copying train pairs...
Copying val pairs...

Unified dataset created at: /kaggle/working/datasets/ardumedics_unified_pose
  Train images: 9332
  Val images: 1646
  Train labels: 9332
  Val labels: 1646


---
## Step 5b: Free Disk Space (Kaggle 20GB Limit)

Delete intermediate extracted frames to free disk space before training.
Kaggle `/kaggle/working/` is limited to **20GB**.

In [7]:
# ============================================================
# STEP 5b: Free disk space by removing intermediate files
# ============================================================

print('Disk usage BEFORE cleanup:')
!df -h /kaggle/working 2>/dev/null || echo '(df not available)'

cleanup_dirs = [
    'ur_fall_frames', 'ur_fall_frames_labels',
    'le2i_frames', 'le2i_frames_labels',
    'multi_cam_frames', 'multi_cam_frames_labels',
    'fall_video_frames', 'fall_video_frames_labels',
    'fall_detection_images', 'fall_detection_images_labels',
    'falling_pose_estimation', 'yolov8_pose_fall',  # ← Fixed: was falling_pose_1/2
    # Raw downloaded datasets (no longer needed after Step 4 auto-labeling)
    'ur_fall_detection', 'le2i_fall',
    'multiple_cameras_fall', 'fall_video_dataset',
    'kaggle_frames',
    'ardumedics_roboflow_temp',  # Clean up temp from Tier 3 download
]

freed_mb = 0
for dir_name in cleanup_dirs:
    dir_path = os.path.join(str(DATASETS_DIR), dir_name)  # ← Fixed: was DATASET_DIR
    if os.path.exists(dir_path):
        total_size = 0
        for root, dirs, files in os.walk(dir_path):
            for f in files:
                fp = os.path.join(root, f)
                if os.path.exists(fp):
                    total_size += os.path.getsize(fp)
        freed_mb += total_size / (1024 * 1024)
        shutil.rmtree(dir_path, ignore_errors=True)
        print(f'  Deleted {dir_name}/ ({total_size/(1024*1024):.1f} MB)')

print(f'\nTotal freed: {freed_mb:.1f} MB')
print('\nDisk usage AFTER cleanup:')
!df -h /kaggle/working 2>/dev/null || echo '(df not available)'
print('\n✓ Step 5b complete: Disk space freed for training')

Disk usage BEFORE cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  7.9G   12G  41% /kaggle/working
  Deleted fall_detection_images/ (50.1 MB)
  Deleted fall_detection_images_labels/ (0.3 MB)
  Deleted falling_pose_estimation/ (36.4 MB)
  Deleted yolov8_pose_fall/ (15.2 MB)
  Deleted ur_fall_detection/ (4278.9 MB)
  Deleted kaggle_frames/ (0.0 MB)

Total freed: 4381.0 MB

Disk usage AFTER cleanup:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  3.6G   16G  18% /kaggle/working

✓ Step 5b complete: Disk space freed for training


## Step 6: Create data.yaml and Verify

Create the `data.yaml` configuration file required by YOLOv8 for training. Verify that the dataset is properly formatted — all images have corresponding labels and annotation formats are correct.

In [8]:
# Notebook: 02 | Step: 6 of 8 — Create data.yaml and Verify
# After this: Train YOLOv8s-Pose model

yaml_path = unified_dir / "data.yaml"

data_config = {
    "path": str(unified_dir),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "fall",
        1: "not-fallen"
    },
    "kpt_shape": [17, 3],  # 17 COCO keypoints, each with (x, y, visibility)
    "flip_idx": [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15],
}

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"data.yaml created at: {yaml_path}")
print(yaml.dump(data_config, default_flow_style=False, sort_keys=False))

# --- Verify dataset integrity ---
print("=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

train_img_dir = unified_dir / "images" / "train"
val_img_dir = unified_dir / "images" / "val"
train_lbl_dir = unified_dir / "labels" / "train"
val_lbl_dir = unified_dir / "labels" / "val"

train_images = sorted(train_img_dir.glob('*'))
val_images = sorted(val_img_dir.glob('*'))
train_labels = sorted(train_lbl_dir.glob('*'))
val_labels = sorted(val_lbl_dir.glob('*'))

print(f"Train: {len(train_images)} images, {len(train_labels)} labels")
print(f"Val:   {len(val_images)} images, {len(val_labels)} labels")

# Check for missing labels
missing_labels = 0
for img in train_images + val_images:
    lbl = (img.parent.parent.parent / "labels" / img.parent.name / (img.stem + ".txt"))
    if not lbl.exists():
        missing_labels += 1

print(f"Images with missing labels: {missing_labels}")

# Verify label format (sample check)
sample_labels = train_labels[:5]
print("\nSample label check:")
for lbl_file in sample_labels:
    with open(lbl_file, 'r') as f:
        line = f.readline().strip()
        parts = line.split()
        # Expected: class(1) + bbox(4) + keypoints(17*3=51) = 56 values
        expected_count = 1 + 4 + 17 * 3
        actual_count = len(parts)
        status = "OK" if actual_count == expected_count else f"WARN (got {actual_count}, expected {expected_count})"
        print(f"  {lbl_file.name}: {actual_count} values [{status}]")

if missing_labels == 0:
    print("\nDataset verification PASSED!")
else:
    print(f"\nDataset verification WARNING: {missing_labels} images missing labels")

data.yaml created at: /kaggle/working/datasets/ardumedics_unified_pose/data.yaml
path: /kaggle/working/datasets/ardumedics_unified_pose
train: images/train
val: images/val
names:
  0: fall
  1: not-fallen
kpt_shape:
- 17
- 3
flip_idx:
- 0
- 2
- 1
- 4
- 3
- 6
- 5
- 8
- 7
- 10
- 9
- 12
- 11
- 14
- 13
- 16
- 15

DATASET VERIFICATION
Train: 9332 images, 9332 labels
Val:   1646 images, 1646 labels
Images with missing labels: 0

Sample label check:
  adl-01-cam0-rgb_adl-01-cam0-rgb-009.txt: 39 values [WARN (got 39, expected 56)]
  adl-01-cam0-rgb_adl-01-cam0-rgb-010.txt: 39 values [WARN (got 39, expected 56)]
  adl-01-cam0-rgb_adl-01-cam0-rgb-011.txt: 39 values [WARN (got 39, expected 56)]
  adl-01-cam0-rgb_adl-01-cam0-rgb-012.txt: 39 values [WARN (got 39, expected 56)]
  adl-01-cam0-rgb_adl-01-cam0-rgb-013.txt: 39 values [WARN (got 39, expected 56)]

Dataset verification PASSED!


## Step 7: Train YOLOv8s-Pose Model

Train the **YOLOv8s-pose (Small)** model with the unified dataset. Key training decisions:

- **AdamW optimizer** with cosine annealing LR schedule for stable convergence
- **100 epochs** — fewer than nano (150) due to larger model capacity
- **Batch size 16** — reduced from 16 (nano) to fit the larger model on T4 GPU memory
- **Extended augmentation** — higher mixup (0.15) and copy-paste (0.3) for regularization
- **Early stopping patience 30** — gives the model enough room to find optimal weights
- **Save checkpoint every 20 epochs** for rollback capability

In [9]:
# Notebook: 02 | Step: 7 of 8 — Train YOLOv8s-Pose Model
# After this: Evaluate model and export to deployment formats

# Load YOLOv8s-pose (Small) pre-trained on COCO keypoints
model = YOLO("yolov8s-pose.pt")
print(f"Model: YOLOv8s-pose")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# Training configuration
TRAIN_CONFIG = {
    'data': str(yaml_path),
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,  # T4x2 DDP: 16 total (8 per GPU) for Small model
    'patience': 30,
    'device': [0, 1],  # T4x2 Dual GPU (DDP)
    'seed': 42,
    'workers': 4,  # Kaggle has 4 CPU cores; 8 causes DataLoader crashes with dual GPU
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'cos_lr': True,
    'warmup_epochs': 5,
    'augment': True,
    'mosaic': 1.0,
    'mixup': 0.15,          # Slightly higher mixup for larger model
    'copy_paste': 0.3,
    'degrees': 15.0,
    'translate': 0.2,
    'scale': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'project': str(RUNS_DIR),
    'name': 'ardumedics_small_pose',
    'exist_ok': True,
    'pretrained': True,
    'verbose': True,
    'val': True,
    'plots': True,
    'save': True,
    'save_period': 20,
}

print("\nTraining Configuration:")
print("-" * 40)
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")
print("-" * 40)

# Start training
print(f"\nStarting YOLOv8s-Pose training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Estimated time: ~10-11 hours on T4 GPU")

results = model.train(**TRAIN_CONFIG)

print(f"\nTraining completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Model: YOLOv8s-pose
Parameters: 11,626,046

Training Configuration:
----------------------------------------
  data: /kaggle/working/datasets/ardumedics_unified_pose/data.yaml
  epochs: 100
  imgsz: 640
  batch: 16
  patience: 30
  device: [0, 1]
  seed: 42
  workers: 4
  optimizer: AdamW
  lr0: 0.001
  lrf: 0.01
  cos_lr: True
  warmup_epochs: 5
  augment: True
  mosaic: 1.0
  mixup: 0.15
  copy_paste: 0.3
  degrees: 15.0
  translate: 0.2
  scale: 0.5
  fliplr: 0.5
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  project: /kaggle/working/runs
  name: ardumedics_small_pose
  exist_ok: True
  pretrained: True
  verbose: True
  val: True
  plots: True
  save: True
  save_period: 20
----------------------------------------

Starting YOLOv8s-Pose training at 2026-05-24 21:38:39
Estimated time: ~10-11 hours on T4 GPU
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: 

## Step 8: Evaluate and Export

Evaluate the trained model on validation and test sets, then export to deployment formats:

- **NCNN** — Optimized for edge devices (Raspberry Pi, Jetson)
- **TFLite (INT8)** — Quantized for mobile/Android deployment
- **ONNX** — Cross-platform inference

Metrics are saved to `metrics_notebook02_small.json` for comparison across notebooks.

In [10]:
# Notebook: 02 | Step: 8 of 8 — Evaluate and Export
# After this: Proceed to Notebook 03 (YOLOv8m-Pose Medium)

# Load best trained model
best_model_path = RUNS_DIR / "ardumedics_small_pose" / "weights" / "best.pt"
print(f"Loading best model from: {best_model_path}")

best_model = YOLO(str(best_model_path))

# --- Evaluate on validation set ---
print("\n" + "=" * 60)
print("VALIDATION SET EVALUATION")
print("=" * 60)

val_metrics = best_model.val(
    data=str(yaml_path),
    split="val",
    imgsz=640,
    batch=8,
    device=0,
    verbose=True,
    plots=True,
)

print(f"\nValidation Results:")
print(f"  mAP50:    {val_metrics.box.map50:.4f}")
print(f"  mAP50-95: {val_metrics.box.map:.4f}")
print(f"  Precision: {val_metrics.box.mp:.4f}")
print(f"  Recall:    {val_metrics.box.mr:.4f}")

# --- Evaluate on test split (using val as test proxy) ---
print("\n" + "=" * 60)
print("TEST SET EVALUATION (val split as proxy)")
print("=" * 60)

test_metrics = best_model.val(
    data=str(yaml_path),
    split="val",
    imgsz=640,
    batch=8,
    device=0,
    verbose=True,
)

print(f"\nTest Results:")
print(f"  mAP50:    {test_metrics.box.map50:.4f}")
print(f"  mAP50-95: {test_metrics.box.map:.4f}")
print(f"  Precision: {test_metrics.box.mp:.4f}")
print(f"  Recall:    {test_metrics.box.mr:.4f}")

# --- Save metrics to JSON (standardized nested format) ---
metrics_dict = {
    "notebook": "02_YOLOv8s_Pose",
    "model": "yolov8s-pose",
    "model_size": "s",
    "best_mAP50": float(val_metrics.box.map50),
    "parameters": "11.2M",
    "epochs": TRAIN_CONFIG['epochs'],
    "batch_size": TRAIN_CONFIG['batch'],
    "imgsz": TRAIN_CONFIG['imgsz'],
    "optimizer": TRAIN_CONFIG['optimizer'],
    "lr0": TRAIN_CONFIG['lr0'],
    "seed": SEED,
    "timestamp": datetime.now().isoformat(),
    "val": {
        "box_precision": float(val_metrics.box.mp),
        "box_recall": float(val_metrics.box.mr),
        "box_map50": float(val_metrics.box.map50),
        "box_map": float(val_metrics.box.map),
        "pose_map50": float(val_metrics.pose.map50),
        "pose_map": float(val_metrics.pose.map),
    },
    "test": {
        "box_precision": float(test_metrics.box.mp),
        "box_recall": float(test_metrics.box.mr),
        "box_map50": float(test_metrics.box.map50),
        "box_map": float(test_metrics.box.map),
        "pose_map50": float(test_metrics.pose.map50),
        "pose_map": float(test_metrics.pose.map),
    },
}

metrics_path = WORK_DIR / "metrics_notebook02_small.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print(f"\nMetrics saved to: {metrics_path}")

# --- Export to deployment formats ---
print("\n" + "=" * 60)
print("MODEL EXPORT")
print("=" * 60)

export_formats = {
    "ncnn": {
        "format": "ncnn",
        "imgsz": 640,
    },
    "tflite_int8": {
        "format": "tflite",
        "imgsz": 640,
        "int8": True,
        "data": str(yaml_path),
    },
    "onnx": {
        "format": "onnx",
        "imgsz": 640,
        "opset": 17,
        "simplify": True,
    },
}

export_results = {}

for name, config in export_formats.items():
    print(f"\nExporting {name}...")
    try:
        export_path = best_model.export(**config)
        export_results[name] = str(export_path)
        
        # Get file size
        size_mb = os.path.getsize(export_path) / (1024 * 1024)
        print(f"  -> Exported to: {export_path}")
        print(f"  -> Size: {size_mb:.1f} MB")
    except Exception as e:
        print(f"  -> Export failed: {e}")
        export_results[name] = f"FAILED: {e}"

# --- Summary of exported models ---
print("\n" + "=" * 60)
print("EXPORT SUMMARY")
print("=" * 60)

for name, path in export_results.items():
    if not path.startswith("FAILED"):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {name:20s} | {size_mb:8.1f} MB | {path}")
    else:
        print(f"  {name:20s} | FAILED   | {path}")

# Also report the PyTorch model size
pt_size = os.path.getsize(best_model_path) / (1024 * 1024)
print(f"  {'pytorch_best':20s} | {pt_size:8.1f} MB | {best_model_path}")

print("\n" + "=" * 60)
print("NOTEBOOK 02 COMPLETE")
print("=" * 60)
print(f"\nModel: YOLOv8s-Pose (Small)")
print(f"Val mAP50: {metrics_dict['val']['box_map50']:.4f}")
print(f"Val mAP50-95: {metrics_dict['val']['box_map']:.4f}")
print(f"\nNext: Notebook 03 — YOLOv8m-Pose (Medium)")

Loading best model from: /kaggle/working/runs/ardumedics_small_pose/weights/best.pt

VALIDATION SET EVALUATION
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-pose summary (fused): 82 layers, 11,616,111 parameters, 0 gradients, 30.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2611.5±733.8 MB/s, size: 341.9 KB)
val: Scanning /kaggle/working/datasets/ardumedics_unified_pose/labels/val.cache... 1646 images, 18 backgrounds, 1490 corrupt: 100% ━━━━━━━━━━━━ 1646/1646 493.1Mit/s 0.0s
val: /kaggle/working/datasets/ardumedics_unified_pose/images/val/adl-01-cam0-rgb_adl-01-cam0-rgb-017.png: ignoring corrupt image/label: labels require 56 columns each
val: /kaggle/working/datasets/ardumedics_unified_pose/images/val/adl-01-cam0-rgb_adl-01-cam0-rgb-018.png: ignoring corrupt image/label: labels require 56 columns each
val: /kaggle/working/datasets/ardumedics_unified_pose/images/val/adl-01-cam0-rgb_adl-01-cam0-rgb-019.png: ignoring corrupt image

pnnxparam = /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.block.C2f
in

NCNN: export success ✅ 10.7s, saved as '/kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model' (44.5 MB)

Export complete (11.3s)
Results saved to /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model
Predict:         yolo predict task=pose model=/kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=pose model=/kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model imgsz=640 data=/kaggle/working/datasets/ardumedics_unified_pose/data.yaml  
Visualize:       https://netron.app
  -> Exported to: /kaggle/working/runs/ardumedics_small_pose/weights/best_ncnn_model
  -> Size: 0.0 MB

Exporting tflite_int8...
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)

PyTorch: starting from '/kaggle/working/runs/ardumedics_small_pose/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 57, 8400) (22.4 MB)
TensorFlow SavedModel: collecting INT8 calibr

E0000 00:00:1779660131.987125      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779660132.034317      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779660132.420811      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779660132.420834      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779660132.420836      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779660132.420839      22 computation_placer.cc:177] computation placer already registered. Please check linka

requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 2.62s
 Downloaded ai-edge-litert
Prepared 5 packages in 751ms
Installed 5 packages in 7ms
 + ai-edge-litert==2.1.5
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.6.1
 + onnx2tf==1.28.8
 + sng4onnx==2.0.1

requirements: AutoUpdate success ✅ 3.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /kaggle/working/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 49.0files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...


I0000 00:00:1779660163.969651      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12907 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779660163.974797      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1779660168.003680      22 cuda_dnn.cc:529] Loaded cuDNN version 91002


Saved artifact at '/kaggle/working/runs/ardumedics_small_pose/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 57, 8400), dtype=tf.float32, name=None)
Captures:
  135359802182352: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  135359802181200: TensorSpec(shape=(3, 3, 3, 32), dtype=tf.float32, name=None)
  135359802181584: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  135359802186576: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  135359802185808: TensorSpec(shape=(3, 3, 32, 64), dtype=tf.float32, name=None)
  135359802183696: TensorSpec(shape=(64,), dtype=tf.float32, name=None)
  135359802186192: TensorSpec(shape=(1, 1, 64, 64), dtype=tf.float32, name=None)
  135359802186768: TensorSpec(shape=(64,), dtype=tf.float32, name=None)
  135359802187920: TensorSpec(shape=(4,), dtype=tf.int64, 

I0000 00:00:1779660177.756452      22 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1779660177.756673      22 single_machine.cc:374] Starting new session
I0000 00:00:1779660177.770049      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12907 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779660177.771482      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
W0000 00:00:1779660179.932817      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779660179.932855      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1779660181.722281      22 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1779660181.722

W0000 00:00:1779660191.359169      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779660191.359197      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1779660191.426964      22 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1779660488.529348      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779660488.529402      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1779660785.211295      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779660785.211324      22 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
W0000 00:00:1779661376.327504      22 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_for

TensorFlow SavedModel: export success ✅ 1849.8s, saved as '/kaggle/working/runs/ardumedics_small_pose/weights/best_saved_model' (146.6 MB)

TensorFlow Lite: starting export with tensorflow 2.19.0...
TensorFlow Lite: export success ✅ 0.0s, saved as '/kaggle/working/runs/ardumedics_small_pose/weights/best_saved_model/best_int8.tflite' (11.6 MB)

Export complete (1850.3s)
Results saved to /kaggle/working/runs/ardumedics_small_pose/weights/best_saved_model/best_int8.tflite
Predict:         yolo predict task=pose model=/kaggle/working/runs/ardumedics_small_pose/weights/best_saved_model/best_int8.tflite imgsz=640 int8
Validate:        yolo val task=pose model=/kaggle/working/runs/ardumedics_small_pose/weights/best_saved_model/best_int8.tflite imgsz=640 data=/kaggle/working/datasets/ardumedics_unified_pose/data.yaml int8 
Visualize:       https://netron.app
  -> Exported to: /kaggle/working/runs/ardumedics_small_pose/weights/best_saved_model/best_int8.tflite
  -> Size: 11.6 MB

Exporting onnx

---
## Step 9: Save Outputs for Next Notebooks

Pack all outputs (trained model, metrics JSON, exports) into a directory
that can be saved as a **Kaggle Dataset** for use by Notebooks 05 and 06.

**Instructions**: After this notebook completes, create a new Kaggle Dataset
from the output. Name it `ardumedics-nb02-small-outputs`.

In [11]:
# ============================================================
# STEP 9: Save outputs for next notebooks
# Pack everything into /kaggle/working/nb02_outputs/ for Kaggle Dataset
# ============================================================

OUTPUT_PACK_DIR = '/kaggle/working/nb02_outputs'
os.makedirs(OUTPUT_PACK_DIR, exist_ok=True)

import shutil

# Copy best model weights
best_pt_src = str(WORK_DIR / 'runs' / 'ardumedics_small_pose' / 'weights' / 'best.pt')
if not os.path.exists(best_pt_src):
    # Try alternate path from training output
    best_pt_src = '/kaggle/working/runs/ardumedics_small_pose/weights/best.pt'
if os.path.exists(best_pt_src):
    shutil.copy2(best_pt_src, f'{OUTPUT_PACK_DIR}/best_small.pt')
    print(f'  Copied best_small.pt ({os.path.getsize(f"{OUTPUT_PACK_DIR}/best_small.pt")/1e6:.1f} MB)')

# Copy metrics JSON
metrics_src = str(WORK_DIR / 'metrics_notebook02_small.json')
if not os.path.exists(metrics_src):
    metrics_src = '/kaggle/working/metrics_notebook02_small.json'
if os.path.exists(metrics_src):
    shutil.copy2(metrics_src, f'{OUTPUT_PACK_DIR}/metrics_notebook02_small.json')
    print('  Copied metrics_notebook02_small.json')

# Copy exported models (if any)
export_dir = str(WORK_DIR / 'exports')
if not os.path.exists(export_dir):
    export_dir = '/kaggle/working/exports'
if os.path.exists(export_dir):
    for item in os.listdir(export_dir):
        src = os.path.join(export_dir, item)
        dst = os.path.join(OUTPUT_PACK_DIR, item)
        if os.path.isdir(src):
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
    print('  Copied export files')

print(f'\nAll outputs packed to: {OUTPUT_PACK_DIR}')
print('\n>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<')
print('    1. Click "Save Version" to run the notebook')
print('    2. After completion, go to Output tab')
print('    3. Create a new Dataset from the nb02_outputs/ folder')
print('    4. Name it: ardumedics-nb02-small-outputs')
print('    5. Add this dataset as input to NB05 and NB06')
print('\n✓ Step 9 complete: Outputs packed for next notebooks')

  Copied best_small.pt (23.5 MB)
  Copied metrics_notebook02_small.json
  Copied export files

All outputs packed to: /kaggle/working/nb02_outputs

>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<
    1. Click "Save Version" to run the notebook
    2. After completion, go to Output tab
    3. Create a new Dataset from the nb02_outputs/ folder
    4. Name it: ardumedics-nb02-small-outputs
    5. Add this dataset as input to NB05 and NB06

✓ Step 9 complete: Outputs packed for next notebooks
